# Pacha Takiy · Espectrograma con capas (alternativa a Sonic Visualiser)

Funciona en el navegador con Google Colab, sin instalar nada. Descarga los audios y las capas desde GitHub y muestra lo mismo que Sonic Visualiser:
- el **espectrograma**;
- los **golpes** marcados;
- los **momentos o temporadas** con su valor;
- la curva del **rumor**;
- en El Niño, la curva del **ONI**.

**Cómo usarlo:** menú **Entorno de ejecución → Ejecutar todas**. Para elegir otro audio, cambia `PISTA` en la primera celda.

In [ ]:
# 1) Elige la pista: "ab_demo", "tarde", "clases" o "temporadas" (El Niño)
PISTA = "temporadas"
BASE = "https://raw.githubusercontent.com/AZXIM/pacha-takiy-acustica/main/sonic-visualiser/"   # tu repositorio público

In [ ]:
# 2) Descargar el audio y sus capas
import io, csv, urllib.request, wave
import numpy as np
def bajar(nombre):
    with urllib.request.urlopen(BASE + nombre) as r:
        return r.read()
wav = wave.open(io.BytesIO(bajar(f"{PISTA}.wav")))
fs = wav.getframerate()
x = np.frombuffer(wav.readframes(wav.getnframes()), dtype=np.int16).astype(float) / 32768
leer = lambda n: [f for f in csv.reader(io.StringIO(bajar(n).decode("utf-8"))) if f]
golpes = leer(f"{PISTA}_golpes.csv")
momentos = leer(f"{PISTA}_momentos.csv")
rumor = np.array([[float(a), float(b)] for a, b in leer(f"{PISTA}_rumor.csv")])
oni = np.array([[float(a), float(b)] for a, b in leer("temporadas_oni.csv")]) if PISTA == "temporadas" else None
print(f"{PISTA}.wav: {len(x)/fs:.1f} s, {len(golpes)} golpes, {len(momentos)} momentos")

In [ ]:
# 3) Escuchar
from IPython.display import Audio
Audio(x, rate=fs)

In [ ]:
# 4) Ver: espectrograma + golpes + momentos + rumor (+ ONI)
import matplotlib.pyplot as plt
filas = 3 if oni is not None else 2
fig, ax = plt.subplots(filas, 1, figsize=(15, 3.2 * filas), sharex=True, gridspec_kw={"height_ratios": [2.2] + [1] * (filas - 1)})
ax[0].specgram(x, NFFT=1024, Fs=fs, noverlap=768, cmap="inferno", vmin=-125, vmax=-35)
ax[0].set_yscale("log"); ax[0].set_ylim(100, 6000); ax[0].set_ylabel("Hz")
ax[0].set_title(f"Pacha Takiy · {PISTA}: espectrograma (golpes de 440 Hz, rumor alrededor de 1200 Hz)", loc="left")
for t, etiqueta in golpes:
    ax[0].axvline(float(t), color="#4cc3b9", lw=0.8, ls="--")
ax[1].plot(rumor[:, 0], rumor[:, 1], color="#e5a940")
ax[1].set_ylabel("rumor (dBFS)")
for t, v, d, etiqueta in momentos:
    t, d = float(t), float(d)
    ax[1].axvspan(t, t + d, color="#e5a940", alpha=0.08)
    if len(momentos) <= 12:
        ax[1].text(t, ax[1].get_ylim()[1], etiqueta, fontsize=8, va="top")
if oni is not None:
    ax[2].plot(oni[:, 0] + 0.75, oni[:, 1], "o-", color="#e4572e", ms=3)
    ax[2].axhline(1.5, ls=":", color="#b3261e"); ax[2].axhline(-0.5, ls=":", color="#2f6fb5")
    ax[2].set_ylabel("ONI (°C) · El Niño")
ax[-1].set_xlabel("segundos")
plt.tight_layout(); plt.show()

In [ ]:
# 5) Comprobación automática: ¿cada golpe de la regla está en el audio?
# Algoritmo de Goertzel a 440 Hz en cuadros de 5 ms (el mismo método de scripts/analisis_acustico.py)
N, H = int(0.005 * fs), int(0.0025 * fs)
ven = np.hanning(N); k = 2 * np.cos(2 * np.pi * 440 / fs)
amp, tiempos = [], []
for i0 in range(0, len(x) - N, H):
    s1 = s2 = 0.0
    for v in x[i0:i0 + N] * ven:
        s1, s2 = v + k * s1 - s2, s1
    amp.append(np.sqrt(max(s1 * s1 + s2 * s2 - k * s1 * s2, 0))); tiempos.append((i0 + N / 2) / fs)
amp = np.array(amp); umbral = np.median(amp) + 0.5 * (amp.max() - np.median(amp))
inicios, ultimo = [], -1
for i in range(1, len(amp)):
    if amp[i - 1] < umbral <= amp[i] and tiempos[i] - ultimo > 0.06:
        inicios.append(tiempos[i]); ultimo = tiempos[i]
regla = [float(t) for t, _ in golpes]
errores = [min(abs(g - r) for r in regla) for g in inicios]
print(f"Golpes de la regla: {len(regla)} · detectados en el audio: {len(inicios)} · error máximo: {1000 * max(errores):.1f} ms")
print("✔ Cada golpe es el dato" if len(regla) == len(inicios) and max(errores) < 0.012 else "✖ Revisar")